Definição da Estrutura

In [0]:
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType, DateType, LongType, TimestampType
 
df = spark.read.table("project_apis.bronze.districts_ibge_raw")

df_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("nome", StringType(), True),
        StructField("municipio", StructType([
            StructField("nome", StringType(), True),
            StructField("microrregiao", StructType([
                StructField("nome", StringType(), True),
                StructField("mesorregiao", StructType([
                    StructField("nome", StringType(), True),
                    StructField("UF", StructType([
                        StructField("nome", StringType(), True),
                        StructField("sigla", StringType(), True),
                        StructField("regiao", StructType([
                            StructField("nome", StringType(), True),
                            StructField("sigla", StringType(), True),
                        ]), True)
                    ]), True)
                ]), True)
            ]), True)
        ]), True)
    ])
)

print(df_schema)



Consolidação da Estrutura base com o Schema

In [0]:
df1 = df.withColumn("distritos_array",functions.from_json(functions.col("raw_json"), df_schema))
df1.show()
print(df1)
df1.printSchema

Tratamento da Estrutura e Criação da Tabela

In [0]:
df2 = df1.withColumn("distritos", functions.explode(functions.col("distritos_array")))
#df2.show()
#df2.printSchema()

df_split = df2.select(
    functions.col("distritos.id").cast(LongType()).alias("id"),
    functions.col("distritos.nome").cast(StringType()).alias("nome"),
    functions.col("distritos.municipio.nome").cast(StringType()).alias("nome_municipio"),
    functions.col("distritos.municipio.microrregiao.nome").cast(StringType()).alias("nome_microrregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.nome").cast(StringType()).alias("nome_mesorregiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.nome").cast(StringType()).alias("nome_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.sigla").cast(StringType()).alias("sigla_UF"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.nome").cast(StringType()).alias("nome_regiao"),
    functions.col("distritos.municipio.microrregiao.mesorregiao.UF.regiao.sigla").cast(StringType()).alias("sigla_regiao"),
    functions.col("ingestion_timestamp").cast(TimestampType()),
    functions.col("source").cast(StringType())
)

df_split.show()
#df_split.printSchema()
